# Pancreas multi-study label transfer (scBIOT 1.2.0)

The prepared benchmark object is part of the scBIOT Figshare collection. Its `bench_split` column defines reference and query cells, so this notebook never relies on an undefined `semi_cell_type`.

- Collection: [AnnData for scBIOT analysis](https://figshare.com/articles/dataset/Anndata_for_scBIOT_analysis/30671669)
- Direct file: [Pancreas.h5ad](https://ndownloader.figshare.com/files/66901160)


In [ ]:
from pathlib import Path
import os
import urllib.request
import numpy as np
import scanpy as sc
import scbiot as scb

RANDOM_STATE = 0
ROOT = Path(os.environ.get("SCBIOT_TUTORIALS_PATH", Path.cwd())).resolve()
if ROOT.name == "R":
    ROOT = ROOT.parent
DATA_DIR = Path(os.environ.get("SCBIOT_TUTORIAL_DATA", ROOT / "inputs")).resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch(filename, url):
    path = DATA_DIR / filename
    if path.exists():
        return path
    partial = path.with_suffix(path.suffix + ".part")
    print(f"Downloading {url} -> {path}")
    urllib.request.urlretrieve(url, partial)
    partial.replace(path)
    return path

def subsample(adata, default_max):
    # Raw is not needed here and can prevent indexed reads from backed sparse files.
    if getattr(adata, "isbacked", False) and adata.raw is not None:
        adata.raw = None
    max_cells = int(os.environ.get("SCBIOT_TUTORIAL_MAX_CELLS", default_max))
    if max_cells > 0 and adata.n_obs > max_cells:
        rng = np.random.default_rng(RANDOM_STATE)
        keep = np.sort(rng.choice(adata.n_obs, max_cells, replace=False))
        return adata[keep].to_memory() if getattr(adata, "isbacked", False) else adata[keep].copy()
    return adata.to_memory() if getattr(adata, "isbacked", False) else adata.copy()

AE_EPOCHS = int(os.environ.get("SCBIOT_AE_EPOCHS", "30"))
USE_GPU = os.environ.get("SCBIOT_USE_GPU", "0") == "1"

def top_variable_genes(adata, layer="counts", n=2000):
    from scipy import sparse
    X = adata.layers[layer] if layer in adata.layers else adata.X
    if sparse.issparse(X):
        mean = np.asarray(X.mean(axis=0)).ravel()
        mean_sq = np.asarray(X.multiply(X).mean(axis=0)).ravel()
        variance = mean_sq - mean * mean
    else:
        variance = np.asarray(X).var(axis=0)
    take = np.argsort(variance)[-min(n, adata.n_vars):]
    return adata.var_names[take].tolist()


In [ ]:
path = fetch("Pancreas.h5ad", "https://ndownloader.figshare.com/files/66901160")
adata = subsample(sc.read_h5ad(path), 16_382)
adata.obs["cell_type"] = adata.obs["celltype"].astype(str)
adata.obs["batch"] = adata.obs["tech"].astype(str)
adata.obs["semi_cell_type"] = np.where(
    adata.obs["bench_split"].astype(str).eq("reference"),
    adata.obs["cell_type"].astype(str),
    "Unknown",
)
if "counts" not in adata.layers:
    adata.layers["counts"] = adata.X.copy()
adata.obs[["batch", "cell_type", "semi_cell_type"]].head()


## Embed, integrate, and transfer labels


In [ ]:
adata = scb.pp.autoencoder(
    adata,
    input_key="counts",
    out_key="X_ae",
    batch_key="batch",
    label_key="semi_cell_type",
    unlabeled_category="Unknown",
    genes=top_variable_genes(adata, n=2000),
    n_top_genes=2000,
    latent_dim=30,
    max_epochs=AE_EPOCHS,
    early_stop_patience=5,
    random_state=RANDOM_STATE,
)
adata, metrics = scb.ot.integrate(
    adata,
    obsm_key="X_ae",
    batch_key="batch",
    out_key="X_supbiot",
    label_key="semi_cell_type",
    unlabeled_category="Unknown",
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
adata = scb.ot.supbiot(
    adata,
    use_rep="X_supbiot",
    input_rep_key="X_ae",
    label_key="semi_cell_type",
    unlabeled_category="Unknown",
    pred_label_key="pred_cell_type",
    pred_conf_key="pred_confidence",
    min_conf=0.0,
    random_state=RANDOM_STATE,
    use_gpu=USE_GPU,
)
metrics


## Evaluate query cells


In [ ]:
from sklearn.metrics import normalized_mutual_info_score
mask = adata.obs["semi_cell_type"].astype(str).eq("Unknown")
score = normalized_mutual_info_score(
    adata.obs.loc[mask, "cell_type"].astype(str),
    adata.obs.loc[mask, "pred_cell_type"].astype(str),
)
print(f"Held-out NMI: {score:.3f}")
sc.pp.neighbors(adata, use_rep="X_supbiot", random_state=RANDOM_STATE)
sc.tl.umap(adata, random_state=RANDOM_STATE)
sc.pl.umap(adata, color=["batch", "cell_type", "pred_cell_type"])
